In [ ]:
import pandas as pd
import re
import os
import kagglehub
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geopy]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# download the fast food restaurants dataset from Kaggle
kaggle_path = kagglehub.dataset_download("datafiniti/fast-food-restaurants")

# find the CSV in the downloaded folder
csv_files = [f for f in os.listdir(kaggle_path) if f.endswith('.csv')]
file_path = os.path.join(kaggle_path, csv_files[0])
print(f"Loaded: {csv_files[0]}")

df = pd.read_csv(file_path)

# clean the restaurant names
df["name"] = df["name"].astype(str).str.strip()

# make a zipcode column from the postal code column
if "postalCode" in df.columns:
    df["zipcode"] = df["postalCode"].astype(str).str.extract(r"(\d{5})")
elif "postal_code" in df.columns:
    df["zipcode"] = df["postal_code"].astype(str).str.extract(r"(\d{5})")
else:
    df["zipcode"] = pd.NA

print(f"Loaded {len(df):,} rows, {df['name'].nunique()} unique restaurant names")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Fast_Food_Restaurants_US.csv'

In [ ]:
# set up the geocoder
geolocator = Nominatim(user_agent="restaurant_zip_fill_jupyter")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

In [ ]:
# get a zipcode from a row
def get_zip_from_row(row):
    parts = []

    if pd.notna(row.get("name")):
        parts.append(str(row["name"]).strip())
    if pd.notna(row.get("address")):
        parts.append(str(row["address"]).strip())
    if pd.notna(row.get("city")):
        parts.append(str(row["city"]).strip())
    if pd.notna(row.get("province")):
        parts.append(str(row["province"]).strip())

    parts.append("USA")
    query = ", ".join(parts)

    try:
        location = geocode(query, addressdetails=True)

        if location is None:
            return None

        raw = getattr(location, "raw", {})
        address = raw.get("address", {})

        postcode = address.get("postcode")
        if postcode:
            match = re.search(r"(\d{5})", str(postcode))
            if match:
                return match.group(1)

        display_name = raw.get("display_name", "")
        match = re.search(r"(\d{5})(?:-\d{4})?", str(display_name))
        if match:
            return match.group(1)

        return None

    except Exception:
        return None

In [ ]:
# clean the zipcode column
def clean_zipcode_column(data):
    data = data.copy()
    data["zipcode"] = data["zipcode"].astype("string")
    data["zipcode"] = data["zipcode"].str.extract(r"(\d{5})", expand=False)
    data["zipcode"] = data["zipcode"].fillna("")
    return data

In [ ]:
# process one restaurant
def process_restaurant(data, restaurant_name, pattern, output_file):
    restaurant_df = data[data["name"].str.contains(pattern, case=False, na=False)].copy()

    before_missing = restaurant_df["zipcode"].isna().sum()

    missing_mask = restaurant_df["zipcode"].isna()
    if missing_mask.sum() > 0:
        restaurant_df.loc[missing_mask, "zipcode_geocoded"] = restaurant_df.loc[missing_mask].apply(get_zip_from_row, axis=1)
        restaurant_df["zipcode"] = restaurant_df["zipcode"].fillna(restaurant_df["zipcode_geocoded"])

    restaurant_df = clean_zipcode_column(restaurant_df)

    wanted_cols = [col for col in [
        "name",
        "address",
        "city",
        "province",
        "postalCode",
        "zipcode",
        "latitude",
        "longitude"
    ] if col in restaurant_df.columns]

    restaurant_df = restaurant_df[wanted_cols].copy()
    restaurant_df.to_csv(output_file, index=False)

    summary = {
        "restaurant": restaurant_name,
        "rows": len(restaurant_df),
        "zipcodes_present": (restaurant_df["zipcode"] != "").sum(),
        "zipcodes_missing": (restaurant_df["zipcode"] == "").sum(),
        "states": restaurant_df["province"].nunique() if "province" in restaurant_df.columns else 0,
        "filled_by_geocode": before_missing - (restaurant_df["zipcode"] == "").sum(),
        "output_file": output_file
    }

    return restaurant_df, summary

In [ ]:
# process the 4 restaurants and save to source_data/
panera_df, panera_summary = process_restaurant(
    df,
    "Panera Bread",
    r"Panera",
    "source_data/panera_locations.csv"
)

chipotle_df, chipotle_summary = process_restaurant(
    df,
    "Chipotle Mexican Grill",
    r"Chipotle",
    "source_data/chipotle_locations.csv"
)

five_guys_df, five_guys_summary = process_restaurant(
    df,
    "Five Guys",
    r"Five Guys",
    "source_data/five_guys_locations.csv"
)

panda_df, panda_summary = process_restaurant(
    df,
    "Panda Express",
    r"Panda Express|Panda",
    "source_data/panda_express_locations.csv"
)

In [ ]:
# make one summary table at the end
summary_df = pd.DataFrame([
    panera_summary,
    chipotle_summary,
    five_guys_summary,
    panda_summary
])

print("done")
display(summary_df)

done


,restaurant,rows,zipcodes_present,zipcodes_missing,states,filled_by_geocode,output_file
0,Panera Bread,76,74,2,27,6,panera_locations.csv
1,Chipotle Mexican Grill,34,32,2,16,1,chipotle_locations.csv
2,Five Guys,24,23,1,15,3,five_guys_locations.csv
3,Panda Express,62,60,2,15,0,panda_express_locations.csv
